In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

BRONZE_SCHEMA = "supplysage_bronze"
SILVER_SCHEMA = "supplysage_silver"

TRANSFORM_RUN_ID = f"silver_external_risk_events_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "13_silver_external_risk_events"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

external_bronze_tables = [
    "bronze_ext_gdelt_doc_raw",
    "bronze_ext_nws_alerts_raw",
    "bronze_ext_cisa_kev_raw",
    "bronze_ext_cpsc_recalls_raw",
    "bronze_ext_ofac_sanctions_raw",
    "bronze_ext_sec_company_submissions_raw",
    "bronze_ext_eia_fuel_prices_raw",
    "bronze_ext_ingestion_batch_manifest"
]

print("TRANSFORM_RUN_ID:", TRANSFORM_RUN_ID)
print("BRONZE_SCHEMA:", BRONZE_SCHEMA)
print("SILVER_SCHEMA:", SILVER_SCHEMA)
print("NOTEBOOK_NAME:", NOTEBOOK_NAME)

# Use latest clean external ingestion batch from the manifest.
latest_manifest_row = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_ingestion_batch_manifest")
    .filter(F.col("status") == "INGESTED")
    .orderBy(F.col("created_at").desc())
    .select("ingestion_batch_id", "created_at", "source_count", "status")
    .first()
)

if latest_manifest_row is None:
    raise ValueError("No INGESTED batch found in bronze_ext_ingestion_batch_manifest.")

INGESTION_BATCH_ID = latest_manifest_row["ingestion_batch_id"]

print("Using INGESTION_BATCH_ID:", INGESTION_BATCH_ID)
print("Batch created_at:", latest_manifest_row["created_at"])
print("Source count:", latest_manifest_row["source_count"])

external_inventory_rows = []

for table_name in external_bronze_tables:
    full_table_name = f"{BRONZE_SCHEMA}.{table_name}"

    try:
        df = spark.table(full_table_name)

        if "ingestion_batch_id" in df.columns:
            row_count = df.filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID).count()
        else:
            row_count = df.count()

        status = "FOUND"

        external_inventory_rows.append({
            "table_name": table_name,
            "full_table_name": full_table_name,
            "row_count": row_count,
            "status": status
        })

        print(f"{table_name}: {row_count} rows")

    except Exception as e:
        external_inventory_rows.append({
            "table_name": table_name,
            "full_table_name": full_table_name,
            "row_count": 0,
            "status": "MISSING"
        })

        print(f"{table_name}: MISSING")


external_inventory_schema = T.StructType([
    T.StructField("table_name", T.StringType(), True),
    T.StructField("full_table_name", T.StringType(), True),
    T.StructField("row_count", T.LongType(), True),
    T.StructField("status", T.StringType(), True)
])

external_inventory_df = spark.createDataFrame(external_inventory_rows, schema=external_inventory_schema)

display(external_inventory_df.orderBy("table_name"))

missing_table_count = external_inventory_df.filter(F.col("status") == "MISSING").count()

print("Missing external Bronze tables:", missing_table_count)

if missing_table_count == 0:
    print("✅ Chunk 1 PASSED: All external Bronze tables are available.")
else:
    print("❌ Chunk 1 FAILED: Some external Bronze tables are missing.")

In [0]:
# Notebook 13, Chunk 2: Build normalized Silver external risk event DataFrames

def add_external_event_metadata(df):
    return (
        df
        .withColumn("ingestion_batch_id", F.lit(INGESTION_BATCH_ID))
        .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
        .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
        .withColumn("silver_created_by", F.lit(CREATED_BY))
        .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
    )


# -----------------------------
# 1. GDELT news articles
# -----------------------------
gdelt_article_schema = T.StructType([
    T.StructField("articles", T.ArrayType(T.StructType([
        T.StructField("url", T.StringType(), True),
        T.StructField("title", T.StringType(), True),
        T.StructField("seendate", T.StringType(), True),
        T.StructField("domain", T.StringType(), True),
        T.StructField("language", T.StringType(), True),
        T.StructField("sourcecountry", T.StringType(), True),
        T.StructField("socialimage", T.StringType(), True)
    ])), True)
])

gdelt_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_gdelt_doc_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

gdelt_events_df = (
    gdelt_raw_df
    .withColumn("parsed_payload", F.from_json(F.col("raw_payload_json"), gdelt_article_schema))
    .withColumn("article", F.explode_outer(F.col("parsed_payload.articles")))
    .filter(F.col("article.url").isNotNull())
    .select(
        F.lit("gdelt").alias("source_name"),
        F.lit("news_disruption").alias("risk_category"),
        F.lit("supply_chain_news").alias("event_type"),
        F.to_timestamp(F.col("article.seendate"), "yyyyMMdd'T'HHmmss'Z'").alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("article.seendate"), "yyyyMMdd'T'HHmmss'Z'")).alias("event_date"),
        F.col("article.title").alias("event_title"),
        F.col("article.title").alias("event_summary"),
        F.col("article.url").alias("source_url"),
        F.col("article.domain").alias("source_domain"),
        F.col("article.sourcecountry").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.col("article.language").alias("language"),
        F.lit("medium").alias("severity"),
        F.lit("article").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

gdelt_events_df = add_external_event_metadata(gdelt_events_df)


# -----------------------------
# 2. NWS weather alerts
# -----------------------------
nws_schema = T.StructType([
    T.StructField("features", T.ArrayType(T.StructType([
        T.StructField("id", T.StringType(), True),
        T.StructField("properties", T.StructType([
            T.StructField("event", T.StringType(), True),
            T.StructField("severity", T.StringType(), True),
            T.StructField("urgency", T.StringType(), True),
            T.StructField("certainty", T.StringType(), True),
            T.StructField("areaDesc", T.StringType(), True),
            T.StructField("headline", T.StringType(), True),
            T.StructField("description", T.StringType(), True),
            T.StructField("instruction", T.StringType(), True),
            T.StructField("sent", T.StringType(), True),
            T.StructField("effective", T.StringType(), True),
            T.StructField("onset", T.StringType(), True),
            T.StructField("expires", T.StringType(), True),
            T.StructField("senderName", T.StringType(), True)
        ]), True)
    ])), True)
])

nws_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_nws_alerts_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

nws_events_df = (
    nws_raw_df
    .withColumn("parsed_payload", F.from_json(F.col("raw_payload_json"), nws_schema))
    .withColumn("feature", F.explode_outer(F.col("parsed_payload.features")))
    .filter(F.col("feature.properties.event").isNotNull())
    .select(
        F.lit("nws").alias("source_name"),
        F.lit("weather_logistics").alias("risk_category"),
        F.col("feature.properties.event").alias("event_type"),
        F.to_timestamp(F.col("feature.properties.onset")).alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("feature.properties.onset"))).alias("event_date"),
        F.col("feature.properties.headline").alias("event_title"),
        F.col("feature.properties.description").alias("event_summary"),
        F.col("feature.id").alias("source_url"),
        F.col("feature.properties.senderName").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.col("feature.properties.areaDesc").alias("event_region"),
        F.lit("en").alias("language"),
        F.lower(F.col("feature.properties.severity")).alias("severity"),
        F.lit("weather_alert").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

nws_events_df = add_external_event_metadata(nws_events_df)


# -----------------------------
# 3. CISA KEV vulnerabilities
# -----------------------------
cisa_schema = T.StructType([
    T.StructField("vulnerabilities", T.ArrayType(T.StructType([
        T.StructField("cveID", T.StringType(), True),
        T.StructField("vendorProject", T.StringType(), True),
        T.StructField("product", T.StringType(), True),
        T.StructField("vulnerabilityName", T.StringType(), True),
        T.StructField("dateAdded", T.StringType(), True),
        T.StructField("shortDescription", T.StringType(), True),
        T.StructField("requiredAction", T.StringType(), True),
        T.StructField("dueDate", T.StringType(), True),
        T.StructField("knownRansomwareCampaignUse", T.StringType(), True)
    ])), True)
])

cisa_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_cisa_kev_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

cisa_events_df = (
    cisa_raw_df
    .withColumn("parsed_payload", F.from_json(F.col("raw_payload_json"), cisa_schema))
    .withColumn("vulnerability", F.explode_outer(F.col("parsed_payload.vulnerabilities")))
    .filter(F.col("vulnerability.cveID").isNotNull())
    .select(
        F.lit("cisa").alias("source_name"),
        F.lit("cyber").alias("risk_category"),
        F.lit("known_exploited_vulnerability").alias("event_type"),
        F.to_timestamp(F.col("vulnerability.dateAdded")).alias("event_timestamp"),
        F.to_date(F.col("vulnerability.dateAdded")).alias("event_date"),
        F.concat_ws(" - ", F.col("vulnerability.cveID"), F.col("vulnerability.vulnerabilityName")).alias("event_title"),
        F.col("vulnerability.shortDescription").alias("event_summary"),
        F.lit("https://www.cisa.gov/known-exploited-vulnerabilities-catalog").alias("source_url"),
        F.lit("cisa.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.when(F.lower(F.col("vulnerability.knownRansomwareCampaignUse")) == "known", F.lit("high"))
         .otherwise(F.lit("medium")).alias("severity"),
        F.lit("cyber_catalog").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

cisa_events_df = add_external_event_metadata(cisa_events_df)


# -----------------------------
# 4. CPSC recalls
# -----------------------------
cpsc_schema = T.ArrayType(T.StructType([
    T.StructField("RecallNumber", T.StringType(), True),
    T.StructField("RecallDate", T.StringType(), True),
    T.StructField("Title", T.StringType(), True),
    T.StructField("Description", T.StringType(), True),
    T.StructField("URL", T.StringType(), True)
]))

cpsc_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_cpsc_recalls_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

cpsc_events_df = (
    cpsc_raw_df
    .withColumn("parsed_payload", F.from_json(F.col("raw_payload_json"), cpsc_schema))
    .withColumn("recall", F.explode_outer(F.col("parsed_payload")))
    .filter(F.col("recall.RecallNumber").isNotNull())
    .select(
        F.lit("cpsc").alias("source_name"),
        F.lit("quality_recall").alias("risk_category"),
        F.lit("product_recall").alias("event_type"),
        F.to_timestamp(F.col("recall.RecallDate")).alias("event_timestamp"),
        F.to_date(F.col("recall.RecallDate")).alias("event_date"),
        F.col("recall.Title").alias("event_title"),
        F.col("recall.Description").alias("event_summary"),
        F.col("recall.URL").alias("source_url"),
        F.lit("saferproducts.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.lit("medium").alias("severity"),
        F.lit("product_recall").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

cpsc_events_df = add_external_event_metadata(cpsc_events_df)


# -----------------------------
# 5. OFAC sanctions files, document-level v0
# -----------------------------
ofac_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_ofac_sanctions_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

ofac_events_df = (
    ofac_raw_df
    .select(
        F.lit("ofac").alias("source_name"),
        F.lit("sanctions_compliance").alias("risk_category"),
        F.col("endpoint_name").alias("event_type"),
        F.to_timestamp(F.col("fetched_at")).alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("fetched_at"))).alias("event_date"),
        F.concat(F.lit("OFAC sanctions file snapshot: "), F.col("endpoint_name")).alias("event_title"),
        F.substring(F.col("raw_payload_json"), 1, 1000).alias("event_summary"),
        F.col("request_url").alias("source_url"),
        F.lit("treasury.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.lit("high").alias("severity"),
        F.lit("sanctions_snapshot").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

ofac_events_df = add_external_event_metadata(ofac_events_df)


# -----------------------------
# 6. SEC EDGAR company submissions, company-level v0
# -----------------------------
sec_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_sec_company_submissions_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

sec_events_df = (
    sec_raw_df
    .select(
        F.lit("sec_edgar").alias("source_name"),
        F.lit("financial_filing").alias("risk_category"),
        F.lit("company_filings_snapshot").alias("event_type"),
        F.to_timestamp(F.col("fetched_at")).alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("fetched_at"))).alias("event_date"),
        F.concat(F.lit("SEC company submissions snapshot: "), F.col("company_name")).alias("event_title"),
        F.concat(
            F.lit("SEC filings metadata captured for "),
            F.col("company_name"),
            F.lit(" ("),
            F.col("ticker"),
            F.lit(", CIK "),
            F.col("cik"),
            F.lit(").")
        ).alias("event_summary"),
        F.col("request_url").alias("source_url"),
        F.lit("sec.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.lit("medium").alias("severity"),
        F.lit("filing_metadata").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

sec_events_df = add_external_event_metadata(sec_events_df)


# -----------------------------
# 7. EIA fuel price snapshot, document-level v0
# -----------------------------
eia_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_eia_fuel_prices_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

eia_events_df = (
    eia_raw_df
    .select(
        F.lit("eia").alias("source_name"),
        F.lit("fuel_logistics_cost").alias("risk_category"),
        F.lit("fuel_price_snapshot").alias("event_type"),
        F.to_timestamp(F.col("fetched_at")).alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("fetched_at"))).alias("event_date"),
        F.lit("EIA fuel price data snapshot").alias("event_title"),
        F.lit("Fuel price data captured for logistics cost risk signals.").alias("event_summary"),
        F.col("request_url").alias("source_url"),
        F.lit("eia.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.lit("medium").alias("severity"),
        F.lit("fuel_price_snapshot").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

eia_events_df = add_external_event_metadata(eia_events_df)


silver_external_risk_events_df = (
    gdelt_events_df
    .unionByName(nws_events_df)
    .unionByName(cisa_events_df)
    .unionByName(cpsc_events_df)
    .unionByName(ofac_events_df)
    .unionByName(sec_events_df)
    .unionByName(eia_events_df)
    .withColumn(
        "external_event_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("source_name"),
                F.col("risk_category"),
                F.col("event_type"),
                F.coalesce(F.col("event_title"), F.lit("")),
                F.coalesce(F.col("source_url"), F.lit("")),
                F.coalesce(F.col("event_date").cast("string"), F.lit(""))
            ),
            256
        )
    )
)

external_event_count = silver_external_risk_events_df.count()

print("Silver external risk event count:", external_event_count)

display(
    silver_external_risk_events_df
    .groupBy("source_name", "risk_category")
    .agg(F.count("*").alias("event_count"))
    .orderBy("source_name", "risk_category")
)

display(
    silver_external_risk_events_df
    .select(
        "external_event_id",
        "source_name",
        "risk_category",
        "event_type",
        "event_date",
        "event_title",
        "severity",
        "source_domain",
        "event_country",
        "event_region"
    )
    .limit(30)
)

print("✅ Chunk 2 complete: Normalized Silver external risk events dataframe created.")

In [0]:
# Notebook 13, Chunk 3: Validate normalized Silver external risk events before writing

external_event_validation_rows = []

total_event_count = silver_external_risk_events_df.count()

external_event_validation_rows.append({
    "validation_check": "external_event_count_positive",
    "expected_value": "> 0",
    "actual_value": str(total_event_count),
    "status": "PASS" if total_event_count > 0 else "FAIL",
    "issue_detail": None if total_event_count > 0 else "No external risk events were created."
})

required_sources = ["gdelt", "nws", "cisa", "cpsc", "ofac", "sec_edgar", "eia"]

source_coverage_df = (
    silver_external_risk_events_df
    .groupBy("source_name")
    .agg(F.count("*").alias("event_count"))
)

actual_sources = [row["source_name"] for row in source_coverage_df.collect()]
missing_sources = [source for source in required_sources if source not in actual_sources]

external_event_validation_rows.append({
    "validation_check": "required_source_coverage",
    "expected_value": ",".join(required_sources),
    "actual_value": ",".join(sorted(actual_sources)),
    "status": "PASS" if len(missing_sources) == 0 else "FAIL",
    "issue_detail": None if len(missing_sources) == 0 else "Missing sources: " + ", ".join(missing_sources)
})

null_required_field_count = (
    silver_external_risk_events_df
    .filter(
        F.col("external_event_id").isNull()
        | F.col("source_name").isNull()
        | F.col("risk_category").isNull()
        | F.col("event_type").isNull()
        | F.col("event_title").isNull()
        | F.col("severity").isNull()
        | F.col("evidence_type").isNull()
    )
    .count()
)

external_event_validation_rows.append({
    "validation_check": "required_fields_not_null",
    "expected_value": "0 null required field rows",
    "actual_value": str(null_required_field_count),
    "status": "PASS" if null_required_field_count == 0 else "FAIL",
    "issue_detail": None if null_required_field_count == 0 else "Nulls found in required external event fields."
})

duplicate_event_id_count = (
    silver_external_risk_events_df
    .groupBy("external_event_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

external_event_validation_rows.append({
    "validation_check": "external_event_id_unique",
    "expected_value": "0 duplicate event ids",
    "actual_value": str(duplicate_event_id_count),
    "status": "PASS" if duplicate_event_id_count == 0 else "FAIL",
    "issue_detail": None if duplicate_event_id_count == 0 else "Duplicate external_event_id values found."
})

invalid_severity_count = (
    silver_external_risk_events_df
    .filter(~F.col("severity").isin("low", "medium", "high", "severe", "extreme", "minor", "moderate", "unknown"))
    .count()
)

external_event_validation_rows.append({
    "validation_check": "severity_values_valid",
    "expected_value": "Known severity values",
    "actual_value": str(invalid_severity_count),
    "status": "PASS" if invalid_severity_count == 0 else "FAIL",
    "issue_detail": None if invalid_severity_count == 0 else "Unexpected severity values found."
})

missing_payload_lineage_count = (
    silver_external_risk_events_df
    .filter(
        F.col("source_payload_hash").isNull()
        | F.col("api_run_id").isNull()
        | F.col("ingestion_batch_id").isNull()
        | F.col("silver_transform_run_id").isNull()
    )
    .count()
)

external_event_validation_rows.append({
    "validation_check": "lineage_fields_not_null",
    "expected_value": "0 missing lineage rows",
    "actual_value": str(missing_payload_lineage_count),
    "status": "PASS" if missing_payload_lineage_count == 0 else "FAIL",
    "issue_detail": None if missing_payload_lineage_count == 0 else "Missing lineage fields found."
})

event_date_null_count = (
    silver_external_risk_events_df
    .filter(F.col("event_date").isNull())
    .count()
)

external_event_validation_rows.append({
    "validation_check": "event_date_parse_check",
    "expected_value": "0 null event_date rows preferred",
    "actual_value": str(event_date_null_count),
    "status": "PASS" if event_date_null_count == 0 else "WARN",
    "issue_detail": None if event_date_null_count == 0 else "Some events have null event_date after parsing."
})

external_event_validation_schema = T.StructType([
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

external_event_validation_df = spark.createDataFrame(
    external_event_validation_rows,
    schema=external_event_validation_schema
)

display(external_event_validation_df.orderBy("status", "validation_check"))

fail_count = external_event_validation_df.filter(F.col("status") == "FAIL").count()
warn_count = external_event_validation_df.filter(F.col("status") == "WARN").count()

print("Silver external event validation failures:", fail_count)
print("Silver external event validation warnings:", warn_count)

In [0]:
duplicate_event_ids_df = (
    silver_external_risk_events_df
    .groupBy("external_event_id")
    .agg(F.count("*").alias("duplicate_count"))
    .filter(F.col("duplicate_count") > 1)
)

display(duplicate_event_ids_df)

display(
    silver_external_risk_events_df
    .join(duplicate_event_ids_df.select("external_event_id"), on="external_event_id", how="inner")
    .select(
        "external_event_id",
        "source_name",
        "risk_category",
        "event_type",
        "event_date",
        "event_title",
        "source_url",
        "source_payload_hash",
        "api_run_id"
    )
    .orderBy("source_name", "event_title")
)

In [0]:
# Notebook 13, Chunk 3A: Deduplicate exact external risk events before validation

before_dedupe_count = silver_external_risk_events_df.count()

silver_external_risk_events_df = (
    silver_external_risk_events_df
    .dropDuplicates([
        "source_name",
        "risk_category",
        "event_type",
        "event_date",
        "event_title",
        "source_url",
        "source_payload_hash",
        "api_run_id"
    ])
    .drop("external_event_id")
    .withColumn(
        "external_event_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("source_name"),
                F.col("risk_category"),
                F.col("event_type"),
                F.coalesce(F.col("event_title"), F.lit("")),
                F.coalesce(F.col("source_url"), F.lit("")),
                F.coalesce(F.col("event_date").cast("string"), F.lit("")),
                F.coalesce(F.col("source_payload_hash"), F.lit("")),
                F.coalesce(F.col("api_run_id"), F.lit(""))
            ),
            256
        )
    )
)

after_dedupe_count = silver_external_risk_events_df.count()

duplicate_event_id_count = (
    silver_external_risk_events_df
    .groupBy("external_event_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

print("External events before dedupe:", before_dedupe_count)
print("External events after dedupe:", after_dedupe_count)
print("Rows removed:", before_dedupe_count - after_dedupe_count)
print("Duplicate external_event_id count after dedupe:", duplicate_event_id_count)

In [0]:
# Notebook 13, Chunk 4A: Create slim external events and raw evidence documents

TARGET_EVENTS_TABLE = f"{SILVER_SCHEMA}.silver_external_risk_events"
TARGET_DOCUMENTS_TABLE = f"{SILVER_SCHEMA}.silver_external_evidence_documents"

# Keep structured event fields, but do NOT repeat huge raw_payload_json on every exploded event.
silver_external_risk_events_slim_df = (
    silver_external_risk_events_df
    .drop("raw_payload_json")
)

# Store raw API payloads once per API run / payload hash.
raw_external_sources = [
    "bronze_ext_gdelt_doc_raw",
    "bronze_ext_nws_alerts_raw",
    "bronze_ext_cisa_kev_raw",
    "bronze_ext_cpsc_recalls_raw",
    "bronze_ext_ofac_sanctions_raw",
    "bronze_ext_sec_company_submissions_raw",
    "bronze_ext_eia_fuel_prices_raw"
]

raw_document_dfs = []

for table_name in raw_external_sources:
    df = (
        spark.table(f"{BRONZE_SCHEMA}.{table_name}")
        .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
        .filter(F.col("response_ok") == True)
        .select(
            "api_run_id",
            "ingestion_batch_id",
            "source_name",
            "endpoint_name",
            "request_url",
            "response_status_code",
            "response_ok",
            "fetched_at",
            "payload_hash",
            "raw_payload_json"
        )
    )

    raw_document_dfs.append(df)

silver_external_evidence_documents_df = raw_document_dfs[0]

for df in raw_document_dfs[1:]:
    silver_external_evidence_documents_df = silver_external_evidence_documents_df.unionByName(df)

silver_external_evidence_documents_df = (
    silver_external_evidence_documents_df
    .dropDuplicates(["api_run_id", "payload_hash"])
    .withColumn(
        "evidence_document_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("source_name"),
                F.col("endpoint_name"),
                F.col("api_run_id"),
                F.col("payload_hash")
            ),
            256
        )
    )
    .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("silver_created_by", F.lit(CREATED_BY))
    .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
)

print("Slim external event rows:", silver_external_risk_events_slim_df.count())
print("Raw evidence document rows:", silver_external_evidence_documents_df.count())

display(
    silver_external_risk_events_slim_df
    .groupBy("source_name", "risk_category")
    .agg(F.count("*").alias("event_count"))
    .orderBy("source_name", "risk_category")
)

display(
    silver_external_evidence_documents_df
    .groupBy("source_name")
    .agg(F.count("*").alias("document_count"))
    .orderBy("source_name")
)

In [0]:
# Notebook 13, Chunk 4B: Write slim events and raw evidence documents

if fail_count == 0:
    (
        silver_external_risk_events_slim_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("source_name", "risk_category")
        .saveAsTable(TARGET_EVENTS_TABLE)
    )

    print(f"✅ Wrote Silver external risk events table: {TARGET_EVENTS_TABLE}")
    print("Event rows written:", silver_external_risk_events_slim_df.count())

    (
        silver_external_evidence_documents_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("source_name")
        .saveAsTable(TARGET_DOCUMENTS_TABLE)
    )

    print(f"✅ Wrote Silver external evidence documents table: {TARGET_DOCUMENTS_TABLE}")
    print("Evidence document rows written:", silver_external_evidence_documents_df.count())

else:
    raise ValueError("Silver external risk event validation failed. Fix issues before writing.")

In [0]:
# Notebook 13, Chunk 5: Read-back verification

written_events_df = spark.table(TARGET_EVENTS_TABLE)
written_documents_df = spark.table(TARGET_DOCUMENTS_TABLE)

written_event_count = written_events_df.count()
written_document_count = written_documents_df.count()

print("Written Silver external risk event rows:", written_event_count)
print("Written Silver external evidence document rows:", written_document_count)

display(
    written_events_df
    .groupBy("source_name", "risk_category")
    .agg(F.count("*").alias("event_count"))
    .orderBy("source_name", "risk_category")
)

display(
    written_documents_df
    .groupBy("source_name")
    .agg(F.count("*").alias("document_count"))
    .orderBy("source_name")
)

if written_event_count == 11508 and written_document_count == 15:
    print("✅ Read-back check passed: Silver external risk events and evidence documents exist with correct row counts.")
else:
    print("❌ Read-back check failed: Row counts do not match expected values.")

In [0]:
# Notebook 13, Chunk 6: Save external risk validation results

split_validation_rows = []

# Validate read-back event count
split_validation_rows.append({
    "validation_check": "written_external_event_row_count",
    "expected_value": "11508",
    "actual_value": str(written_event_count),
    "status": "PASS" if written_event_count == 11508 else "FAIL",
    "issue_detail": None if written_event_count == 11508 else "Written event row count does not match expected 11,508."
})

# Validate read-back document count
split_validation_rows.append({
    "validation_check": "written_evidence_document_row_count",
    "expected_value": "15",
    "actual_value": str(written_document_count),
    "status": "PASS" if written_document_count == 15 else "FAIL",
    "issue_detail": None if written_document_count == 15 else "Written evidence document row count does not match expected 15."
})

# Events should be slim: no raw payload repeated on every event
event_has_raw_payload_column = "raw_payload_json" in written_events_df.columns

split_validation_rows.append({
    "validation_check": "events_table_excludes_raw_payload_json",
    "expected_value": "False",
    "actual_value": str(event_has_raw_payload_column),
    "status": "PASS" if event_has_raw_payload_column == False else "FAIL",
    "issue_detail": None if event_has_raw_payload_column == False else "raw_payload_json found in events table."
})

# Documents should retain raw payload once per API run
document_missing_payload_count = (
    written_documents_df
    .filter(
        F.col("raw_payload_json").isNull()
        | (F.length(F.trim(F.col("raw_payload_json"))) == 0)
    )
    .count()
)

split_validation_rows.append({
    "validation_check": "evidence_documents_have_raw_payload",
    "expected_value": "0 missing raw payload rows",
    "actual_value": str(document_missing_payload_count),
    "status": "PASS" if document_missing_payload_count == 0 else "FAIL",
    "issue_detail": None if document_missing_payload_count == 0 else "Some evidence documents are missing raw_payload_json."
})

# Document IDs should be unique
duplicate_document_id_count = (
    written_documents_df
    .groupBy("evidence_document_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

split_validation_rows.append({
    "validation_check": "evidence_document_id_unique",
    "expected_value": "0 duplicate evidence document ids",
    "actual_value": str(duplicate_document_id_count),
    "status": "PASS" if duplicate_document_id_count == 0 else "FAIL",
    "issue_detail": None if duplicate_document_id_count == 0 else "Duplicate evidence_document_id values found."
})

split_validation_schema = T.StructType([
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

split_validation_df = spark.createDataFrame(split_validation_rows, schema=split_validation_schema)

all_external_validation_df = (
    external_event_validation_df
    .unionByName(split_validation_df)
)

external_validation_results_df = (
    all_external_validation_df
    .withColumn("transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("source_table", F.lit("multiple_bronze_external_api_raw_tables"))
    .withColumn("target_table", F.lit("silver_external_risk_events_and_evidence_documents"))
    .withColumn("validated_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("created_by", F.lit(CREATED_BY))
    .withColumn("notebook_name", F.lit(NOTEBOOK_NAME))
    .select(
        "transform_run_id",
        "source_table",
        "target_table",
        "validation_check",
        "expected_value",
        "actual_value",
        "status",
        "issue_detail",
        "validated_at",
        "created_by",
        "notebook_name"
    )
)

validation_results_table = f"{SILVER_SCHEMA}.silver_transform_validation_results"

try:
    spark.sql(f"""
    DELETE FROM {validation_results_table}
    WHERE transform_run_id = '{TRANSFORM_RUN_ID}'
    """)
except Exception as e:
    print("Validation results table does not exist yet. It will be created now.")

(
    external_validation_results_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(validation_results_table)
)

display(
    spark.table(validation_results_table)
    .filter(F.col("transform_run_id") == TRANSFORM_RUN_ID)
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

final_fail_count = (
    spark.table(validation_results_table)
    .filter(F.col("transform_run_id") == TRANSFORM_RUN_ID)
    .filter(F.col("status") == "FAIL")
    .count()
)

print("Final Notebook 13 validation failures:", final_fail_count)

if final_fail_count == 0:
    print("✅ Notebook 13 PASSED: Silver external risk events and evidence documents created and validated.")
    print("Next notebook: 14_silver_relationship_validation")
else:
    print("❌ Notebook 13 FAILED: Review validation failures before moving on.")